In [5]:
import pandas as pd
from gerar_mapa import BASE
from math import radians, sin, cos, sqrt, atan2

arquivo_origem = BASE / "dados" / "SURVEY_STARNAV.xlsx"
arquivo_saida = BASE / "dados" / "SURVEY_STARNAV_D2.xlsx"

arquivo_posicoes= BASE/ "dados" / "SURVEY_POSICOES.xlsx"

# Lê todas as abas
planilhas = pd.read_excel(arquivo_origem, sheet_name=None)

# Lê a aba com as posições fixas
posicoes = pd.read_excel(arquivo_posicoes, sheet_name="posicoes")

# Mantém apenas linhas válidas
posicoes = posicoes.dropna(subset=["latitude", "longitude"])


with pd.ExcelWriter(arquivo_saida, engine="openpyxl") as writer:
    
    def haversine(lat1, lon1, lat2, lon2):

        R = 6371  # raio da Terra em km

        lat1 = radians(lat1)
        lon1 = radians(lon1)
        lat2 = radians(lat2)
        lon2 = radians(lon2)

        dlat = lat2 - lat1
        dlon = lon2 - lon1

        a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2

        c = 2 * atan2(sqrt(a), sqrt(1-a))

        return R * c


    for nome_aba, df in planilhas.items():
                
        print(f"Processando {nome_aba}")
        
        # Ignora abas sem a coluna data_consulta
        if "data_consulta" not in df.columns:
            print(f"Aba {nome_aba} ignorada.")
            continue

        # Converte para datetime
        df["data_consulta"] = pd.to_datetime(df["data_consulta"],dayfirst=True)
        
        df["data_reportada"] = pd.to_datetime( df["data_reportada"], dayfirst=True)
        
        
        # Calcula diferença
        delta_consulta = df["data_consulta"].diff()
        delta_reportada = df["data_reportada"].diff()

        # Cria um novo dataframe somente com os indicadores
        resultado = pd.DataFrame({
            "data_consulta": df["data_consulta"],
            "delta_dia_consultada": delta_consulta,
            "data_reportada": df["data_reportada"],
            "delta_dia_reportada": delta_reportada,
        })
        
        distancias = [None]
        menor_distancia = [None]
        unidade_proxima = [None]
        tipo_unidade = [None]
        campo_ = [None]
        evidencia_proximidade = [None]
        

        for i in range(1, len(df)):

            d = haversine(
                df.loc[i-1, "lat_a"],
                df.loc[i-1, "lon_a"],
                df.loc[i, "lat_a"],
                df.loc[i, "lon_a"]
            )

            #distancias.append(d)
            distancias.append(round(d, 2))
            
            menor = float("inf")
            unidade = None
            tipo = None
            campo = None

            lat = df.loc[i, "lat_a"]
            lon = df.loc[i, "lon_a"]

            # Procura a unidade mais próxima
            for _, pos in posicoes.iterrows():

                d = haversine(
                    lat,
                    lon,
                    pos["latitude"],
                    pos["longitude"]
                )

                if d < menor:
                    menor = d
                    unidade = pos["unidade"]
                    tipo = pos["tipo"]
                    campo = pos["campo"]

            # --------------------------------
            # Evidência baseada na proximidade
            # --------------------------------

            if menor <= 1:
                evidencia = 1.0

            elif menor <= 3:
                evidencia = 0.8

            elif menor <= 5:
                evidencia = 0.5

            elif menor <= 10:
                evidencia = 0.2

            else:
                evidencia = 0.0

            menor_distancia.append(round(menor, 3))
            unidade_proxima.append(unidade)
            tipo_unidade.append(tipo)
            campo_.append(campo)
            evidencia_proximidade.append(evidencia)

        resultado["distancia_km"] = distancias 

        # Intervalo entre as coletas
        resultado["delta_horas_input"] = (
            resultado["delta_dia_consultada"]
            .dt.total_seconds() / 3600
        )

        # Intervalo entre as posições reportadas pelo AIS
        resultado["delta_horas_reportada"] = (
            resultado["delta_dia_reportada"]
            .dt.total_seconds() / 3600
        )

        # Velocidade estimada entre as coletas
        resultado["velocidade_kmh"] = (
            resultado["distancia_km"] /
            resultado["delta_horas_input"]
        )

        resultado["velocidade_kmh"] = (
            resultado["velocidade_kmh"].round(2)
        )

        resultado["velocidade_knots"] = (
            resultado["velocidade_kmh"] / 1.852
        ).round(2)

        resultado["status de navegação"] = df["status"]

        resultado["menor_distancia_km"] = menor_distancia
        resultado["unidade_proxima"] = unidade_proxima
        resultado["tipo_unidade"] = tipo_unidade
        resultado["campo"] = campo_

        resultado["evidencia_proximidade"] = evidencia_proximidade

        # Tempo entre a coleta e a última posição reportada
        resultado["delta_coleta_reportada"] = (
            resultado["data_consulta"] -
            resultado["data_reportada"]
        )

        # Verificação antes de salvar
        print("\nCOLUNAS DO RESULTADO:")
        print(resultado.columns.tolist())

        print("\nEVIDÊNCIA DE PROXIMIDADE:")
        print(resultado["evidencia_proximidade"].head(10))

        # Salva na mesma aba
        resultado.to_excel(
            writer,
            sheet_name=nome_aba,
            index=False
        )

        print("Arquivo criado com sucesso!")

# Verifica o arquivo gerado
teste = pd.read_excel(
    arquivo_saida,
    sheet_name=nome_aba
)

print("\nCOLUNAS DO EXCEL GERADO:")
print(teste.columns.tolist())

print("\nEVIDÊNCIA LIDA DO EXCEL:")
print(teste["evidencia_proximidade"].head(10))


Processando posicoes
Aba posicoes ignorada.
Processando STARNAV ANDROMEDA

COLUNAS DO RESULTADO:
['data_consulta', 'delta_dia_consultada', 'data_reportada', 'delta_dia_reportada', 'distancia_km', 'delta_horas_input', 'delta_horas_reportada', 'velocidade_kmh', 'velocidade_knots', 'status de navegação', 'menor_distancia_km', 'unidade_proxima', 'tipo_unidade', 'campo', 'evidencia_proximidade', 'delta_coleta_reportada']

EVIDÊNCIA DE PROXIMIDADE:
0    NaN
1    0.8
2    0.8
3    0.8
4    0.8
5    0.2
6    0.8
7    0.8
8    0.2
9    0.2
Name: evidencia_proximidade, dtype: float64
Arquivo criado com sucesso!
Processando STARNAV AQUARIUS

COLUNAS DO RESULTADO:
['data_consulta', 'delta_dia_consultada', 'data_reportada', 'delta_dia_reportada', 'distancia_km', 'delta_horas_input', 'delta_horas_reportada', 'velocidade_kmh', 'velocidade_knots', 'status de navegação', 'menor_distancia_km', 'unidade_proxima', 'tipo_unidade', 'campo', 'evidencia_proximidade', 'delta_coleta_reportada']

EVIDÊNCIA DE PR

C:\Users\roger\AppData\Local\Temp\ipykernel_39908\3755363363.py:51: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["data_consulta"] = pd.to_datetime(df["data_consulta"],dayfirst=True)



COLUNAS DO RESULTADO:
['data_consulta', 'delta_dia_consultada', 'data_reportada', 'delta_dia_reportada', 'distancia_km', 'delta_horas_input', 'delta_horas_reportada', 'velocidade_kmh', 'velocidade_knots', 'status de navegação', 'menor_distancia_km', 'unidade_proxima', 'tipo_unidade', 'campo', 'evidencia_proximidade', 'delta_coleta_reportada']

EVIDÊNCIA DE PROXIMIDADE:
0    NaN
1    0.8
2    1.0
3    0.0
4    0.0
5    0.0
6    0.0
7    0.0
8    0.0
9    0.0
Name: evidencia_proximidade, dtype: float64
Arquivo criado com sucesso!
Processando STARNAV REGULUS

COLUNAS DO RESULTADO:
['data_consulta', 'delta_dia_consultada', 'data_reportada', 'delta_dia_reportada', 'distancia_km', 'delta_horas_input', 'delta_horas_reportada', 'velocidade_kmh', 'velocidade_knots', 'status de navegação', 'menor_distancia_km', 'unidade_proxima', 'tipo_unidade', 'campo', 'evidencia_proximidade', 'delta_coleta_reportada']

EVIDÊNCIA DE PROXIMIDADE:
0    NaN
1    1.0
2    1.0
3    1.0
4    1.0
5    1.0
6    1.0
7

In [7]:
import pandas as pd
from gerar_mapa import BASE

# ==========================
# ARQUIVOS DA FROTA
# ==========================

arquivos = [
    BASE / "dados" / "SURVEY_BRAM_D1.xlsx",
    BASE / "dados" / "SURVEY_CBO_D1.xlsx",
    BASE / "dados" / "SURVEY_STARNAV_D1.xlsx"
]

total_frota = 0

print("=" * 60)

for arquivo in arquivos:

    print(f"\nArquivo: {arquivo.name}")

    planilhas = pd.read_excel(arquivo, sheet_name=None)

    total_empresa = 0

    for nome_aba, df in planilhas.items():

        if "distancia_km" not in df.columns:
            continue

        distancia = df["distancia_km"].sum(skipna=True)

        print(f"{nome_aba:<25} {distancia:10.2f} km")

        total_empresa += distancia

    print("-" * 60)
    print(f"TOTAL {arquivo.stem:<18} {total_empresa:10.2f} km")

    total_frota += total_empresa

print("\n" + "=" * 60)
print(f"TOTAL GERAL DA FROTA: {total_frota:.2f} km")
print("=" * 60)

######################################################################################


#Atualizar os cards da langind page
from pathlib import Path

BASE = Path.cwd().parent      # se estiver executando dentro de src
# ou BASE = Path.cwd()         # se executar na raiz do projeto

script = f"""const ESTATISTICAS = {{
    km: {total_frota},
    empresas: 3,
    psvs: 53,
    campos: 18
}};

document.getElementById("km").dataset.target = ESTATISTICAS.km;
document.getElementById("campos").dataset.target = ESTATISTICAS.campos;
document.getElementById("empresas").dataset.target = ESTATISTICAS.empresas;
document.getElementById("psvs").dataset.target = ESTATISTICAS.psvs;

const counters = document.querySelectorAll(".contador");

counters.forEach(counter => {{

    const target = parseFloat(counter.dataset.target);

    let current = 0;

    const increment = Math.max(1, Math.ceil(target / 80));

    function update(){{

        current += increment;

        if(current >= target){{

            counter.innerText = Math.round(target).toLocaleString("pt-BR");

        }}else{{

            counter.innerText = Math.round(current).toLocaleString("pt-BR");

            requestAnimationFrame(update);

        }}

    }}

    update();

}});
"""

with open(BASE / "script.js", "w", encoding="utf-8") as f:
    f.write(script)

print("script.js atualizado com sucesso.")



Arquivo: SURVEY_BRAM_D1.xlsx
BRAM ATLAS                    593.18 km
BRAM BAHIA                    355.65 km
BRAM BELEM                    585.28 km
BRAM BRASIL                   354.40 km
BRAM BRASILIA                 677.18 km
BRAM BRAVO                    281.45 km
BRAM BREEZE                   658.31 km
BRAM BUCK                     724.07 km
BRAM BUZIOS                   111.97 km
BRAM HERO                    1031.42 km
BRAM POWER                    179.47 km
BRAM RIO                      635.81 km
BRAM SPIRIT                   190.42 km
BRAM TITAN                    499.06 km
------------------------------------------------------------
TOTAL SURVEY_BRAM_D1        6877.67 km

Arquivo: SURVEY_CBO_D1.xlsx
CBO ALESSANDRA                114.90 km
CBO ALIANCA                   591.76 km
CBO ANITA                    1348.11 km
CBO ARPOADOR                 2890.65 km
CBO CAMPOS                      1.10 km
CBO CAROLINA                  494.08 km
CBO COPACABANA                901.38 km
C